# Project 1 — Advanced EDA & Feature Engineering
**Dataset:** `Dataset_for_Data_Analytics.xlsx` (1,200 e-commerce orders)
**Goal:** Clean the raw order data and prepare it for downstream ML use by (1) handling missing values with a defensible statistical method, (2) detecting and treating outliers, and (3) engineering at least three new predictive features.

The notebook follows the standard flow: inspect → diagnose missingness/outliers → treat → engineer features → validate → export.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

df = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

Rows: 1200, Columns: 14


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


## 1. Initial Inspection

Before touching anything, get a baseline: data types, missing values, and summary statistics. This tells us where to focus — there's no point running an imputation strategy on a column that's already complete.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


In [3]:
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct}).query('missing_count > 0')

,missing_count,missing_pct
CouponCode,309,25.75


Only `CouponCode` has missing values — **309 rows, 25.75%**. Every other column is complete, so the imputation work in this project centers entirely on this one field.

## 2. Handling Missing Values — `CouponCode`

Per the missing-data decision matrix, a 25.75% missingness rate sits above the 20% threshold, which is normally where you'd reach for a heavier method like KNN. But before applying any statistical formula, it's worth asking a more basic question: **why** is this value missing?

Let's check what `CouponCode` actually contains.

In [4]:
df['CouponCode'].value_counts(dropna=False)

CouponCode
FREESHIP    313
NaN         309
WINTER15    292
SAVE10      286
Name: count, dtype: int64

This is the key diagnostic. `CouponCode` only ever holds three real values — `SAVE10`, `FREESHIP`, `WINTER15` — plus `NaN`. There's no fourth "no code" category anywhere in the data. That means the missing values almost certainly aren't sensor errors or dropped records — they represent orders where **no coupon was applied at all**. This is a case of **Missing Not At Random (MNAR)**: the missingness itself carries information tied to customer behavior, not a random data-collection gap.

Applying Mean/Median/KNN imputation here would be a mistake — those methods are built for *numeric* columns where you're estimating a plausible underlying value. There is no "true" coupon code hiding behind these NaNs to estimate; the customer simply didn't use one. Treating this as an estimation problem would manufacture fake categories and quietly bias any downstream model that uses `CouponCode` as a predictor.

**For comparison, here's what the two candidate approaches look like:**

In [5]:
# Option A — Statistical imputation (mode / most-frequent-category)
# This is the closest categorical analogue to "Mean/Median" imputation.
mode_value = df['CouponCode'].mode()[0]
coupon_mode_imputed = df['CouponCode'].fillna(mode_value)
print(f"Mode imputation would fill all 309 missing rows with: '{mode_value}'")
print(coupon_mode_imputed.value_counts())

Mode imputation would fill all 309 missing rows with: 'FREESHIP'
CouponCode
FREESHIP    622
WINTER15    292
SAVE10      286
Name: count, dtype: int64


Mode imputation would label all 309 no-coupon orders as `FREESHIP`, which is factually wrong for every single one of them and inflates that category by roughly 35%. That's the same distortion the brief's trade-off table warns about with global-median imputation deflating variance — except here it would fabricate a false coupon usage rate.

In [6]:
# Option B — Business-logic imputation: encode the missingness itself
df['CouponCode'] = df['CouponCode'].fillna('NoCoupon')
df['CouponCode'].value_counts()

CouponCode
FREESHIP    313
NoCoupon    309
WINTER15    292
SAVE10      286
Name: count, dtype: int64

This keeps the data honest (a `NoCoupon` order is not the same as a `FREESHIP` order) and, as a bonus, sets up a clean binary feature later in the feature engineering section. This is the approach carried forward.

## 3. Outlier Detection — Interquartile Range (IQR)

With missingness resolved, the next check is for outliers across all numeric columns. Using the standard IQR rule:

$$\text{Lower Bound} = Q_1 - 1.5 \times IQR \qquad \text{Upper Bound} = Q_3 + 1.5 \times IQR$$

In [7]:
numeric_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice']
outlier_summary = {}

for col in numeric_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = df[(df[col] < lower) | (df[col] > upper)].shape[0]
    outlier_summary[col] = {'lower_bound': round(lower, 2), 'upper_bound': round(upper, 2), 'n_outliers': n_outliers}

pd.DataFrame(outlier_summary).T

,lower_bound,upper_bound,n_outliers
Quantity,-1.00,7.00,0.0
UnitPrice,-317.20,1024.83,0.0
ItemsInCart,-0.50,11.50,0.0
TotalPrice,-1341.41,3330.41,8.0


`Quantity`, `UnitPrice`, and `ItemsInCart` come back completely clean — zero boundary violations. `TotalPrice` flags 8 rows. Rather than capping them on sight, let's inspect what's actually driving them.

In [8]:
Q1, Q3 = df['TotalPrice'].quantile(0.25), df['TotalPrice'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

flagged = df[df['TotalPrice'] > upper][['OrderID', 'Quantity', 'UnitPrice', 'TotalPrice']].copy()
flagged['Quantity_x_UnitPrice'] = flagged['Quantity'] * flagged['UnitPrice']
flagged

,OrderID,Quantity,UnitPrice,TotalPrice,Quantity_x_UnitPrice
107,ORD200107,5,670.75,3353.75,3353.75
326,ORD200326,5,670.48,3352.40,3352.40
328,ORD200328,5,674.04,3370.20,3370.20
469,ORD200469,5,676.98,3384.90,3384.90
632,ORD200632,5,678.16,3390.80,3390.80
789,ORD200789,5,691.28,3456.40,3456.40
1065,ORD201065,5,666.80,3334.00,3334.00
1122,ORD201122,5,678.19,3390.95,3390.95


Every flagged row checks out: `TotalPrice` equals `Quantity x UnitPrice` exactly, and both of those inputs sit well within their own normal ranges (`Quantity` = 5, the max in the dataset; `UnitPrice` in the high-600s, still under its own upper bound of ~1,025). These aren't data-entry glitches or hardware faults — they're legitimate high-ticket orders that happen to combine two normal values into a large product. `TotalPrice` is a *derived* column, so it will naturally look more "spread out" than either of its inputs.

Capping or removing these 8 rows would delete real sales, not noise — the exact risk the brief calls out with row deletion (destroys volume, invites bias). **Decision: leave `TotalPrice` untouched.** The IQR check earns its keep here precisely by giving us a reason *not* to intervene — the goal is a mathematically clean dataset, not a heavily-clipped one.

For completeness, this is the capping logic that *would* apply if these had been genuine anomalies (e.g., a $50M order from a data entry error):

In [9]:

print("No capping applied — all 8 flagged values verified as genuine orders.")

No capping applied — all 8 flagged values verified as genuine orders.


## 4. Feature Engineering

Four new features, each targeting a different signal that the raw columns don't expose directly.

**Feature 1 — `HasCoupon`**
A binary flag derived from the missingness we just handled. Whether a coupon was used at all is often more predictive than *which* coupon — this converts the "no code" state into a clean 0/1 signal instead of leaving it buried inside a categorical column.

In [10]:
df['HasCoupon'] = (df['CouponCode'] != 'NoCoupon').astype(int)
df['HasCoupon'].value_counts()

HasCoupon
1    891
0    309
Name: count, dtype: int64

**Feature 2 — `CartConversionRate`**
`ItemsInCart` (what the customer added) vs. `Quantity` (what they actually bought) tells us how "decisive" a shopper was. A ratio close to 1 means they bought almost everything they browsed; a low ratio suggests a lot of cart abandonment on this specific order.

In [11]:
df['CartConversionRate'] = (df['Quantity'] / df['ItemsInCart']).round(3)
df['CartConversionRate'].describe()

count    1200.000000
mean        0.579878
std         0.249856
min         0.167000
25%         0.400000
50%         0.500000
75%         0.750000
max         1.000000
Name: CartConversionRate, dtype: float64

**Feature 3 — `OrderMonth` and `IsWeekendOrder`**
The raw `Date` column isn't directly usable by most estimators. Extracting month captures seasonality (e.g., holiday shopping spikes); flagging weekend orders separates weekday/weekend purchase behavior, which is a common retail signal.

In [12]:
df['OrderMonth'] = df['Date'].dt.month
df['IsWeekendOrder'] = df['Date'].dt.dayofweek.isin([5, 6]).astype(int)
df[['Date', 'OrderMonth', 'IsWeekendOrder']].head()

,Date,OrderMonth,IsWeekendOrder
0,2023-01-04,1,0
1,2024-08-23,8,0
2,2024-02-27,2,0
3,2023-10-15,10,1
4,2025-05-08,5,0


**Feature 4 — `IsRepeatCustomer`**
11 `CustomerID`s appear twice in this dataset. Whether a customer has ordered before is a standard, high-value retail feature (repeat customers behave differently than first-time ones), and it's cheap to compute via a frequency count.

In [13]:
customer_counts = df['CustomerID'].value_counts()
df['IsRepeatCustomer'] = df['CustomerID'].map(lambda x: 1 if customer_counts[x] > 1 else 0)
df['IsRepeatCustomer'].value_counts()

IsRepeatCustomer
0    1178
1      22
Name: count, dtype: int64

### Checking for multicollinearity among the new numeric features

The brief flags multicollinearity (`corr > 0.80`) as a structural risk — highly correlated predictors make regression coefficients unstable. Quick check before finalizing:

In [14]:
feature_cols = ['Quantity', 'UnitPrice', 'ItemsInCart', 'TotalPrice', 'CartConversionRate', 'OrderMonth', 'HasCoupon', 'IsWeekendOrder', 'IsRepeatCustomer']
corr_matrix = df[feature_cols].corr().round(2)
corr_matrix

,Quantity,UnitPrice,ItemsInCart,TotalPrice,CartConversionRate,OrderMonth,HasCoupon,IsWeekendOrder,IsRepeatCustomer
Quantity,1.00,0.01,0.65,0.62,0.37,-0.02,-0.04,-0.03,-0.02
UnitPrice,0.01,1.00,0.00,0.72,0.02,-0.03,0.04,-0.01,-0.03
ItemsInCart,0.65,0.00,1.00,0.39,-0.41,-0.02,-0.02,-0.02,-0.03
TotalPrice,0.62,0.72,0.39,1.00,0.23,-0.03,0.01,-0.02,-0.03
CartConversionRate,0.37,0.02,-0.41,0.23,1.00,0.02,-0.02,-0.01,0.00
OrderMonth,-0.02,-0.03,-0.02,-0.03,0.02,1.00,0.02,-0.02,-0.04
HasCoupon,-0.04,0.04,-0.02,0.01,-0.02,0.02,1.00,0.02,-0.00
IsWeekendOrder,-0.03,-0.01,-0.02,-0.02,-0.01,-0.02,0.02,1.00,0.01
IsRepeatCustomer,-0.02,-0.03,-0.03,-0.03,0.00,-0.04,-0.00,0.01,1.00


No pair exceeds the 0.80 threshold (the highest is `TotalPrice`/`UnitPrice` at ~0.72, which is expected since `TotalPrice` is partly derived from `UnitPrice`). Nothing needs to be dropped.

## 5. Final Validation

In [15]:
print("Remaining missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0] if df.isnull().sum().sum() > 0 else "None — dataset is complete.")
print()
print("Final shape:", df.shape)
print()
print("New columns added:", ['HasCoupon', 'CartConversionRate', 'OrderMonth', 'IsWeekendOrder', 'IsRepeatCustomer'])
df.head()

Remaining missing values:
None — dataset is complete.

Final shape: (1200, 19)

New columns added: ['HasCoupon', 'CartConversionRate', 'OrderMonth', 'IsWeekendOrder', 'IsRepeatCustomer']


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice,HasCoupon,CartConversionRate,OrderMonth,IsWeekendOrder,IsRepeatCustomer
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10,1,0.714,1,0,0
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70,1,0.667,8,0,0
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40,1,0.625,2,0,0
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19,1,0.200,10,1,0
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04,1,0.500,5,0,0


## 6. Export Cleaned Dataset

In [16]:
df.to_excel('Dataset_for_Data_Analytics_CLEANED.xlsx', index=False)
print("Saved: Dataset_for_Data_Analytics_CLEANED.xlsx")

Saved: Dataset_for_Data_Analytics_CLEANED.xlsx


## Summary

| Requirement | What was done | Why |
|---|---|---|
| Missing values | `CouponCode` (25.75% missing) filled with `'NoCoupon'`, not mean/median/KNN | The missingness is MNAR — it means "no coupon," not "unknown coupon." Statistical imputation would have fabricated false coupon usage. |
| Outliers | IQR applied to all 4 numeric columns; 8 flagged in `TotalPrice` | Investigated and confirmed as legitimate high-ticket orders (`Quantity x UnitPrice` checks out); left untouched rather than capping real sales. |
| Feature engineering | Added `HasCoupon`, `CartConversionRate`, `OrderMonth` + `IsWeekendOrder`, `IsRepeatCustomer` | Each captures a distinct behavioral or seasonal signal not directly present in the raw columns; correlation check confirms no redundancy (multicollinearity) introduced. |

The dataset is now complete (zero missing values), has no unresolved outlier risk, and carries five additional engineered signals — ready to be handed to a modeling pipeline.